# GENETIC RELATIONSHIP MATRIX: STEP 03

This script creates PLINK files from array data in the African, American, and European ancestry groups using variants that passed the second pruning step for GRM calculation.

In [ ]:
# Load packages 
from datetime import datetime
import pandas as pd 
import os

In [ ]:
# Workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")
# Workspace bucket 
my_bucket

# COMMAND 

In [ ]:
%%writefile plink_files_for_grm.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    # Check if the cohort file for the current population exists
    if [ ! -f "${cohort}/${pop}-like_cohort_qc.txt" ]; then
        echo "Cohort file for ${pop} not found!"
        exit 1
    fi
    
    # Copy new plink module into the plink2 
    cp "${plink2}" plink2

    # Grant permision to run plink2 in environment
    chmod +x plink2
    
    # Check number of samples in each cohort 
    echo "Number of samples in ${pop}:"
    wc -l "${cohort}/${pop}-like_cohort_qc.txt"
    
    # PLINK2 command to perform quality control and LD pruning on SNV variants for the current population

    # - Uses a binary PLINK file as input.
    # - Filters out variants with low minor allele frequency (MAF < 0.01).
    # - Removes variants with high missingness (>1% missing genotypes).
    # - Excludes variants failing Hardy-Weinberg equilibrium (p < 1^-6).
    # - Restricts analysis to autosomal chromosomes (1-22).
    # - Retains only individuals specified in the cohort file (defined by a population file).
    # - Performs linkage disequilibrium (LD) pruning within a 50 step window, removing variants with r² > 0.05
    # - Saves the output files to the specified output directory.
    # - Restricts to single-nucleotide variants (SNVs) and biallelic variants only.
    # - Uses multithreading (4 threads) for faster execution.
    
    # Run PLINK2 for quality control on SNV variants for the current population
    ./plink2 --bfile "${genos}/arrays" \
             --keep "${cohort}/${pop}-like_cohort_qc.txt" \
             --extract "${markers}/${pop}-like_grm_pruned.prune.in" \
             --make-bed --out "${output}/${pop}_pruned_for_grm"

done

In [ ]:
# Copy plink_files_for_grm.sh script to my_bucket 
!gsutil cp plink_files_for_grm.sh {my_bucket}/data/scripts/                                    

In [ ]:
# Check plink_files_for_grm.sh script in my_bucket 
!gsutil cat {my_bucket}/data/scripts/plink_files_for_grm.sh 

# SUBMIT COMMAND 

In [ ]:
%%bash --out grm_plink_files

#############################################################################
# DSUB: SUBMIT SCRIPT FOR GENERATING GRM PLINK FILES 
#############################################################################

# Source the dsub setup script, which likely contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/microarray/plink"

#############################################################################
# SUBMIT DSUB SCRIPT JOB FOR GENERATING GRM PLINK FILES 
#############################################################################

# Use a prebuilt container with PLINK2
# Set total disk space required 
# Define logging directory for dsub execution logs
# Input PLINK files (BED, BIM, FAM) from Google Cloud Storage (GCS)
# Provide cohort sample lists (recursively input from GCS)
# Define output directory for variant QC results (recursively stores outputs)
# Set environment variables for the script
# Specify the script to execute within the container

aou_dsub \
  --disk-size 450 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/ancestry_sd3_wgs_qc/" \
  --input-recursive markers="${WORKSPACE_BUCKET}/data/saige_step00/pruned_for_grm/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step00/grm_plink_files/" \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data//wgs_qc/ancestry_sd3_wgs_qc" \
  --env markers="${WORKSPACE_BUCKET}/data/saige_step00/pruned_for_grm" \
  --env output="${WORKSPACE_BUCKET}/data/saige_step00/grm_plink_files" \
  --script "${WORKSPACE_BUCKET}/data/scripts/plink_files_for_grm.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################
